In [ ]:
!pip install textblob


In [ ]:
import pandas as pd
import re
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
df = pd.read_csv('reviews.csv', on_bad_lines='skip')

In [ ]:
# import pandas as pd

# # Attempt to read the CSV, handling bad lines by skipping them
# try:
#     df = pd.read_csv('all_cities_cleaned_reviews.csv', on_bad_lines='skip', engine='python')  # Use the Python engine
# except pd.errors.ParserError as e:
#     print(f"Error reading CSV: {e}")
#     # If 'skip' doesn't resolve the issue, consider manually inspecting and fixing
#     # the CSV file (e.g., using a text editor) to correct the problematic line.
#     print("Please check for unterminated string around line 91314 of the CSV file.")

FileNotFoundError: [Errno 2] No such file or directory: 'all_cities_cleaned_reviews.csv'

In [ ]:
df.head()

In [ ]:
# Combine comments for the same listing_id
# df = df.groupby('listing_id')['comments'].apply(lambda x: ' '.join(x)).reset_index()
df = df.groupby('listing_id')['comments'].apply(
    lambda x: ' '.join(x.dropna().astype(str))
).reset_index()


In [ ]:
df['listing_id'].nunique()

16444

In [ ]:
df.columns

Index(['listing_id', 'comments'], dtype='object')

In [ ]:
# Function to clean text
def clean_text(text):
    text = re.sub(r'[^\w\s]', '', text)  # Remove punctuation
    text = re.sub(r'\d+', '', text)  # Remove numbers
    return text.lower()  # Convert to lowercase

In [ ]:
df = df[df['comments'].str.strip().notna() & (df['comments'].str.strip() != '') & (df['comments'] != 'nan')]
df['comments'] = df['comments'].astype(str).apply(lambda x: x.strip())


<ipython-input-96-bea3774b016f>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['comments'] = df['comments'].astype(str).apply(lambda x: x.strip())


In [ ]:
df['comments'] = df['comments'].astype(str).apply(clean_text)

In [ ]:
df['comments'].isnull().sum()

np.int64(0)

In [ ]:
# Initialize CountVectorizer with stop words removal
custom_stop_words = ['great', 'place', 'stay', 'good', 'best','able']

french_stop_words = stopwords.words('french')
stop_words = list(ENGLISH_STOP_WORDS) + custom_stop_words + list(french_stop_words)

# stop_words = list(ENGLISH_STOP_WORDS) + custom_stop_words
vectorizer = CountVectorizer(stop_words=stop_words, max_features=500)  # Limit vocab size to 500

# Fit on full dataset, but process row-by-row
vectorizer.fit(df['comments'])



CountVectorizer(max_features=500,
                stop_words=['my', 'whose', 'or', 'than', 'over', 'first', 'his',
                            'un', 'both', 'latterly', 'fire', 'whereafter',
                            'almost', 'detail', 'i', 'somewhere', 'now',
                            'cannot', 'formerly', 'had', 'toward', 'mine',
                            'why', 'myself', 'few', 'whatever', 'done', 'out',
                            'behind', 'call', ...])

In [ ]:
def get_top_words(text):
    words = vectorizer.transform([text]).toarray().flatten()
    word_freq = dict(zip(vectorizer.get_feature_names_out(), words))
    top_words = sorted(word_freq, key=word_freq.get, reverse=True)[:5]
    return ', '.join(top_words)


In [ ]:
# Apply function row-wise
df['top_words'] = df['comments'].apply(get_top_words)


In [ ]:
from textblob import TextBlob

# Function to classify sentiment based on polarity
def get_sentiment(text):
    blob = TextBlob(text)
    polarity = blob.sentiment.polarity
    if polarity > 0.1:
        return 'Positive'
    elif polarity < -0.03:
        return 'Negative'
    else:
        return 'Neutral'



# Apply sentiment analysis on this smaller portion
df['Predicted Sentiment'] = df['comments'].apply(get_sentiment)

# Print the results to see how the sentiment looks
df[['comments', 'Predicted Sentiment']].head()


,comments,Predicted Sentiment
0,having the opportunity of arriving to alexandr...,Positive
1,we had a lovely time in toronto kathie and lar...,Positive
2,my friends and i absolutely loved our stay at ...,Positive
3,i and my family really enjoyed our stay at br...,Positive
4,fabulous accomodation matched only be the view...,Positive


In [ ]:
df.columns

Index(['listing_id', 'comments', 'top_words', 'Predicted Sentiment'], dtype='object')

In [ ]:
# Print the results to see how the sentiment looks
df['Predicted Sentiment'].value_counts()

,count
Predicted Sentiment,
Positive,16079
Neutral,298
Negative,66


In [ ]:
df_sen=df[['listing_id','top_words','Predicted Sentiment']]

In [ ]:
df_sen.head()

,listing_id,top_words,Predicted Sentiment
0,1419,"home, house, beautiful, br, easy",Positive
1,8077,"room, toronto, view, hosts, br",Positive
2,26654,"location, apartment, host, recommend, clean",Positive
3,27423,"host, toronto, apartment, need, br",Positive
4,30931,"canada, experience, fabulous, friendly, home",Positive


In [ ]:
df_sen.to_csv('sentiment_analysis_Toronto.csv', index=False)


In [ ]:
df['Predicted Sentiment'].head(3)

,Predicted Sentiment
0,Positive
1,Positive
2,Positive
